# 08 · Variables para actualizar el pronóstico con producción reciente

**Entra:** base analítica oficial. **Sale:** casos y catálogo en memoria, sin nuevos datasets.

Hipótesis central: lo que acaba de producirse puede anticipar si debemos subir o bajar el pronóstico de las semanas siguientes. Se construyen variables pasadas, no la respuesta futura. H1 es la semana que comienza en el origen; H5 es cuatro semanas después (+1…+5 respecto a la última semana cerrada).

Este mismo bloque explícito de preparación se incluye en 09 para que pueda ejecutarse de forma independiente, sin importar scripts propios ni depender de variables de otro kernel.

## 1. Imports, carga y columnas autorizadas

In [1]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos").is_dir() and (p / "Notebooks").is_dir())
SALIDA = ROOT / "Datos_analiticos_proyecto"
SALIDA.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 35)
print("Fuentes originales:", ROOT / "Datos")
print("Salida oficial:", SALIDA)

base = pd.read_parquet(SALIDA/"base_analitica.parquet")
KEY = ["finca","producto","color","variedad"]
SEM = KEY+["semana_inicio"]
assert not base.duplicated(SEM).any()
print(base.shape)


Fuentes originales: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos
Salida oficial: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto


(44053, 108)


## 2. Preparación temporal explícita

Producción: lags 1–3, medias 4/8, variabilidad y tendencia. Ventanas cortas para estudiar actualización; no se seleccionaron por maximizar una métrica. Referencia de 52 semanas: sólo cuando existe, sin convertir faltantes en cero.

Planos: último estado anterior al origen bajo el supuesto conservador de disponibilidad después del cierre nominal y de AL1, con caducidad de cuatro semanas. AL1 no certifica publicación. La edad avanza desde el plano, sin suponer nuevas siembras o erradicaciones conocidas. La prueba de retraso adicional muestra sensibilidad, no certifica puntualidad.

Clima: lags y medias de cuatro semanas cerradas, exigiendo al menos cinco días por media semanal. Radiación se conserva como promedio de la magnitud fuente, no energía acumulada. Lluvia/evapotranspiración permanecen en el limpio hasta verificar unidad y definición de intervalo.

In [2]:
def construir_casos(base, retraso_semanas=0):
    """Una fila = serie/origen/horizonte. Ninguna X usa la producción objetivo."""
    key = ["finca","producto","color","variedad"]
    b = base.copy()
    b["semana_inicio"] = b.semana_inicio.astype("datetime64[ns]")
    calendario = pd.DataFrame({"origen":pd.date_range(b.semana_inicio.min(),
                        b.semana_inicio.max()+pd.Timedelta(weeks=1),freq="W-MON")})
    calendario["origen"] = calendario.origen.astype("datetime64[ns]")
    grid = b[key].drop_duplicates().merge(calendario,how="cross")
    grid = grid.merge(b[key+["semana_inicio","tallos_reales"]],
                      left_on=key+["origen"],right_on=key+["semana_inicio"],
                      how="left",validate="one_to_one").sort_values(key+["origen"])
    g = grid.groupby(key,sort=False).tallos_reales
    desfase = 1+retraso_semanas
    for lag in [1,2,3]:
        grid[f"produccion_lag{lag}"] = g.shift(lag+retraso_semanas)
    for v in [4,8]:
        grid[f"media_{v}"] = g.transform(lambda s:s.shift(desfase).rolling(v,min_periods=v).mean())
    grid["desv_4"] = g.transform(lambda s:s.shift(desfase).rolling(4,min_periods=4).std())
    grid["tendencia_4_8"] = grid.media_4-grid.media_8
    grid["cambio_reciente"] = grid.produccion_lag1-grid.produccion_lag2
    grid["observaciones_previas"] = g.transform(lambda s:s.notna().cumsum().shift(desfase,fill_value=0))
    grid = grid.drop(columns=["semana_inicio","tallos_reales"])

    # Planos: no llamar publicación a AL1. Supuesto prudente para el experimento:
    # disponibilidad posterior al cierre nominal Y a la fecha usada para calcular edad.
    p = b[key+["semana_inicio","fecha_referencia_edad_plano","plantas_plano","edad_media_plano"]].dropna(subset=["plantas_plano"]).copy()
    p["disponible_plano"] = pd.concat([p.semana_inicio+pd.Timedelta(days=7),
        pd.to_datetime(p.fecha_referencia_edad_plano)+pd.Timedelta(days=1)],axis=1).max(axis=1)
    p["disponible_plano"] = (p.disponible_plano+pd.Timedelta(weeks=retraso_semanas)).astype("datetime64[ns]")
    p = p.sort_values(["disponible_plano","semana_inicio"]).drop_duplicates(key+["disponible_plano"],keep="last")
    p = p.rename(columns={"semana_inicio":"semana_plano_usado","plantas_plano":"plantas_conocidas",
                          "edad_media_plano":"edad_plano_conocida"})
    grid = pd.merge_asof(grid.sort_values("origen"),p.drop(columns="fecha_referencia_edad_plano").sort_values("disponible_plano"),
        left_on="origen",right_on="disponible_plano",by=key,direction="backward",tolerance=pd.Timedelta(weeks=4))
    assert (grid.disponible_plano.isna() | grid.disponible_plano.le(grid.origen)).all()
    grid["antiguedad_plano"] = (grid.origen-grid.semana_plano_usado).dt.days/7
    grid["edad_proyectada_origen"] = grid.edad_plano_conocida+grid.antiguedad_plano

    # Cada finca/semana aparece en muchas variedades, pero clima sólo se cuenta una vez.
    cols = ["temperatura_semana","humedad_semana","radiacion_semana"]
    assert b.groupby(["finca","semana_inicio"])[cols].nunique().le(1).all().all()
    clima = b.groupby(["finca","semana_inicio"])[cols+["dias_clima"]].first().reset_index()
    clima.loc[clima.dias_clima.lt(5),cols] = np.nan  # una semana muy incompleta no representa su promedio
    clima = b[["finca"]].drop_duplicates().merge(calendario,how="cross").merge(
        clima.rename(columns={"semana_inicio":"origen"}),on=["finca","origen"],how="left",validate="one_to_one").sort_values(["finca","origen"])
    for col in cols:
        clima[col+"_lag1"] = clima.groupby("finca")[col].shift(desfase)
        clima[col+"_media4"] = clima.groupby("finca")[col].transform(lambda s:s.shift(desfase).rolling(4,min_periods=4).mean())
    clima = clima.drop(columns=cols+["dias_clima"])
    grid = grid.merge(clima,on=["finca","origen"],how="left",validate="many_to_one")

    etiquetas = b[key+["semana_inicio","tallos_reales"]].rename(columns={"semana_inicio":"semana_objetivo","tallos_reales":"y"})
    casos = []
    for h in range(1,6):
        d = grid.loc[grid.observaciones_previas.gt(0)].copy()
        d["horizonte"] = h
        d["semana_objetivo"] = d.origen+pd.Timedelta(weeks=h-1)
        d["ultima_semana_utilizada"] = d.origen-pd.Timedelta(weeks=desfase)
        d = d.merge(etiquetas,on=key+["semana_objetivo"],how="left",validate="many_to_one")
        d["fecha_estacional"] = d.semana_objetivo-pd.Timedelta(weeks=52)
        estacional = etiquetas.rename(columns={"semana_objetivo":"fecha_estacional","y":"produccion_52"})
        d = d.merge(estacional,on=key+["fecha_estacional"],how="left",validate="many_to_one")
        assert (d.fecha_estacional+pd.Timedelta(weeks=1+retraso_semanas)).le(d.origen).all()
        semana = d.semana_objetivo.dt.isocalendar().week.astype(float)
        d["semana_seno"] = np.sin(2*np.pi*semana/52)
        d["semana_coseno"] = np.cos(2*np.pi*semana/52)
        d["edad_proyectada_objetivo"] = d.edad_proyectada_origen+h-1
        casos.append(d)
    casos = pd.concat(casos,ignore_index=True)
    assert not casos.duplicated(key+["origen","horizonte"]).any()
    assert casos.ultima_semana_utilizada.lt(casos.origen).all()
    return casos

categoricas = ["producto","color","variedad"]
temporales = ["horizonte","semana_seno","semana_coseno"]
memoria = ["produccion_lag1","produccion_lag2","produccion_lag3","media_4","media_8",
           "desv_4","tendencia_4_8","cambio_reciente","produccion_52"]
agronomicas = ["plantas_conocidas","edad_proyectada_objetivo","antiguedad_plano"]
climaticas = [v+s for v in ["temperatura_semana","humedad_semana","radiacion_semana"] for s in ["_lag1","_media4"]]
X_columnas = categoricas+temporales+memoria+agronomicas+climaticas
assert "y" not in X_columnas


## 3. Generar casos y mostrar cobertura

Las filas sin y permanecen para pronosticar o diagnosticar, pero no se puntúan como producción cero. La evaluación principal de 09 exigirá historia reciente completa y al menos 26 observaciones previas; informará cuánto volumen deja fuera.

In [3]:
casos = construir_casos(base)
display(casos.groupby("horizonte").agg(casos=("origen","size"),con_real=("y","count")))
display(casos[X_columnas].isna().mean().mul(100).round(1).to_frame("faltantes_pct"))
display(casos[KEY+["origen","semana_objetivo","horizonte","produccion_lag1","media_4","plantas_conocidas","disponible_plano","y"]].tail(10))


,casos,con_real
horizonte,,
1,124239,43390
2,124239,42800
3,124239,42196
4,124239,41598
5,124239,41017


,faltantes_pct
producto,0.0
color,0.0
variedad,0.0
horizonte,0.0
semana_seno,0.0
semana_coseno,0.0
produccion_lag1,64.5
produccion_lag2,64.7
produccion_lag3,64.8
media_4,68.4


,finca,producto,color,variedad,origen,semana_objetivo,horizonte,produccion_lag1,media_4,plantas_conocidas,disponible_plano,y
621185,GAITANA,BARBATUS,WHITE,ONYX WHITE,2026-09-07,2026-10-05,5,<NA>,NaN,NaN,NaT,<NA>
621186,GAITANA,SOLOMIO,RED,CRIS,2026-09-07,2026-10-05,5,4890,3558.5,18000.0,2026-08-20,<NA>
621187,GAITANA,MINICARNATION,WHITE,6-593,2026-09-07,2026-10-05,5,<NA>,NaN,NaN,NaT,<NA>
621188,GAITANA,CARNATION,GREEN,C 205,2026-09-07,2026-10-05,5,<NA>,NaN,NaN,NaT,<NA>
621189,GAITANA,MINICARNATION,BICOLOR PINK,TEKNO,2026-09-07,2026-10-05,5,<NA>,NaN,NaN,NaT,<NA>
621190,GAITANA,CARNATION,RED,GENARO,2026-09-07,2026-10-05,5,<NA>,NaN,NaN,NaT,<NA>
621191,GAITANA,BARBATUS,WHITE,POETHUS WHITE,2026-09-07,2026-10-05,5,<NA>,NaN,NaN,NaT,<NA>
621192,GAITANA,SOLOMIO,RED,CHERRYFLY,2026-09-07,2026-10-05,5,104,101.0,500.0,2026-08-20,<NA>
621193,GAITANA,CARNATION,LIGHT PINK,CUTE,2026-09-07,2026-10-05,5,<NA>,NaN,NaN,NaT,<NA>
621194,GAITANA,VERONICA SPRAY,WHITE,SKYLER SPLASH WHITE,2026-09-07,2026-10-05,5,2300,2090.0,15040.0,2026-08-20,<NA>


## 4. Catálogo pequeño de variables

Una función no oculta la lógica: arriba se ven las operaciones completas. Las variables se agrupan por pregunta agronómica y se compararán por bloques. No se usan el potencial descriptivo de todo el histórico, ni errores WebFlor sin origen, ni estimados de un archivo posterior como predictores.

In [4]:
catalogo_features = pd.DataFrame([
 ["Identidad",", ".join(categoricas),"Dimensiones comerciales","Conocida al origen; categorías se ajustan en train","Composición por variedad/color"],
 ["Calendario",", ".join(temporales),"Semana objetivo en seno/coseno; horizonte","Conocido sin observar la producción objetivo","Estacionalidad"],
 ["Memoria",", ".join(memoria),"Shift previo; rolling no centrado; diferencia medias","Sólo semanas cerradas; sensibilidad a demora","Producción reciente actualiza semanas siguientes"],
 ["Siembras",", ".join(agronomicas),"Último plano admisible; edad avanza por calendario","Disponibilidad supuesta y antigüedad explícitas","Población/edad explican potencial"],
 ["Clima",", ".join(climaticas),"Media semanal diaria; lag 1 y media pasada 4","Sólo mediciones pasadas; estación/unidades por confirmar","Efectos ambientales rezagados"],
],columns=["familia","variables","calculo","disponibilidad_y_riesgo","hipotesis"])
display(catalogo_features)
print("Las cantidades reales futuras son exclusivamente y.")


,familia,variables,calculo,disponibilidad_y_riesgo,hipotesis
0,Identidad,"producto, color, variedad",Dimensiones comerciales,Conocida al origen; categorías se ajustan en t...,Composición por variedad/color
1,Calendario,"horizonte, semana_seno, semana_coseno",Semana objetivo en seno/coseno; horizonte,Conocido sin observar la producción objetivo,Estacionalidad
2,Memoria,"produccion_lag1, produccion_lag2, produccion_l...",Shift previo; rolling no centrado; diferencia ...,Sólo semanas cerradas; sensibilidad a demora,Producción reciente actualiza semanas siguientes
3,Siembras,"plantas_conocidas, edad_proyectada_objetivo, a...",Último plano admisible; edad avanza por calend...,Disponibilidad supuesta y antigüedad explícitas,Población/edad explican potencial
4,Clima,"temperatura_semana_lag1, temperatura_semana_me...",Media semanal diaria; lag 1 y media pasada 4,Sólo mediciones pasadas; estación/unidades por...,Efectos ambientales rezagados


Las cantidades reales futuras son exclusivamente y.


## 5. Prueba contra uso accidental del futuro

Alteramos producción, clima y planos desde una fecha de prueba. Las X de orígenes anteriores o iguales deben permanecer idénticas. Esto comprueba la implementación temporal, no la fecha real de publicación de los archivos.

In [5]:
fecha_prueba = pd.Timestamp("2026-01-05")
alterada = base.copy()
columnas_prueba = ["tallos_reales","plantas_plano","edad_media_plano",
                  "temperatura_semana","humedad_semana","radiacion_semana"]
alterada.loc[alterada.semana_inicio.ge(fecha_prueba),columnas_prueba] *= 2
prueba = construir_casos(alterada)
llave_caso = KEY+["origen","horizonte"]
a = casos.loc[casos.origen.le(fecha_prueba)].set_index(llave_caso)[[c for c in X_columnas if c not in llave_caso]].sort_index()
z = prueba.loc[prueba.origen.le(fecha_prueba)].set_index(llave_caso)[[c for c in X_columnas if c not in llave_caso]].sort_index()
pd.testing.assert_frame_equal(a,z)
print("PRUEBA APROBADA: cambiar el futuro no cambia las features de orígenes anteriores.")
del alterada,prueba,a,z
CORTE = pd.Timestamp("2026-01-05")
admisible = casos.y.notna() & casos.media_4.notna() & casos.observaciones_previas.ge(26)
train = casos.loc[admisible & (casos.semana_objetivo+pd.Timedelta(days=7)).le(CORTE)]
test = casos.loc[admisible & casos.origen.ge(CORTE)]
print("Ejemplo train:",len(train),"evaluación retrospectiva:",len(test))
print("El entrenamiento y la comparación se ejecutan en 09. Este cuadernillo no exporta otra base.")


PRUEBA APROBADA: cambiar el futuro no cambia las features de orígenes anteriores.


Ejemplo train: 123537 evaluación retrospectiva: 19174
El entrenamiento y la comparación se ejecutan en 09. Este cuadernillo no exporta otra base.


## 6. Decisión para modelar

09 compara baselines, regresión y boosting sobre los mismos casos, con cortes temporales. Una comparación por bloques medirá el aporte de la memoria productiva, siembras y clima. También estudiará una corrección al promedio móvil; no se presentará como corrección WebFlor.

Las fechas de publicación no están certificadas. El experimento es retrospectivo bajo supuestos explícitos y se prueba una demora adicional. Los modelos no deben ponerse en operación sin validar esos supuestos, las unidades y la interpretación de ausencias de producción.